# Customer Health History (Rolling 90-Day Snapshot)

Daily **full snapshot** of the SCD2 `customer_health_profile`'s **open (current) versions** — one row per active lead per day, all 24 version columns + run_date. Complements the SCD2 version history in `customer_health_profile` (what changed, when) with a fixed daily trend log (band transitions, score drift). Maintains a 90-day rolling window. Runs after 3_14's SCD2 MERGE.

In [0]:
%sql
-- Daily health snapshot: append today's OPEN SCD2 versions (current state per
-- lead) to a rolling 90-day history table. customer_health_profile is now SCD2
-- (multiple rows per lead), so ONLY open versions (VRSN_END_DATE IS NULL) are
-- snapshotted — one row per active lead per day. Closed versions live in the
-- profile table itself. Idempotent: deletes today's rows first (safe re-runs).

CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_history (
    run_date DATE,
    -- Identity + versioning
    LEAD_ID STRING,
    LEAD_EMAIL STRING,
    LEAD_NAME STRING,
    VRSN_START_DATE DATE,
    VRSN_END_DATE DATE,
    -- Commercial
    CONTRACT_VALUE DOUBLE,
    CONTRACT_START_DATE DATE,
    -- Engagement metrics
    DAYS_SINCE_LAST_EMAIL INT,
    DAYS_SINCE_LAST_LOGIN INT,
    DAYS_SINCE_LAST_MEETING INT,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT BIGINT,
    DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER BIGINT,
    TOTAL_MISSED_CALLS BIGINT,
    SENTIMENTS_LAST_30_DAYS STRING,
    -- Flags
    SLACK_ENGAGED_FLAG INT,
    PLATFORM_ENGAGED_FLAG INT,
    MEETING_ENGAGED_FLAG INT,
    ENGAGEMENT_PATTERN STRING,
    -- Scoring
    BASE_ENGAGEMENT_SCORE INT,
    FINAL_SCORE INT,
    HEALTH_BAND STRING,
    HASH_VALUE STRING,
    SOURCE_UPDATED_AT TIMESTAMP,
    INSERT_DATE TIMESTAMP
);

DELETE FROM crm_ingestion.gold.customer_health_history WHERE run_date = CURRENT_DATE;

INSERT INTO crm_ingestion.gold.customer_health_history
SELECT
    CURRENT_DATE AS run_date,
    LEAD_ID, LEAD_EMAIL, LEAD_NAME, VRSN_START_DATE, VRSN_END_DATE,
    CONTRACT_VALUE, CONTRACT_START_DATE,
    DAYS_SINCE_LAST_EMAIL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_MEETING,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    TOTAL_MISSED_CALLS, SENTIMENTS_LAST_30_DAYS,
    SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
    ENGAGEMENT_PATTERN,
    BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND,
    HASH_VALUE, SOURCE_UPDATED_AT, INSERT_DATE
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL

In [0]:
%sql
-- Rolling 90-day retention: delete snapshots older than 90 days
DELETE FROM crm_ingestion.gold.customer_health_history
WHERE run_date < DATE_SUB(CURRENT_DATE, 90)

In [0]:
%sql
-- Summary: client count and average score by run_date and health_band
SELECT
    run_date,
    HEALTH_BAND,
    COUNT(*) AS client_count,
    ROUND(AVG(FINAL_SCORE), 1) AS avg_score
FROM crm_ingestion.gold.customer_health_history
GROUP BY run_date, health_band
ORDER BY run_date DESC, health_band

In [0]:
%sql
select * from crm_ingestion.gold.customer_health_history